In [ ]:
import numpy as np
import os
from scipy.fft import fft, fftshift
from tqdm import tqdm
from dotenv import load_dotenv

from rfml_uav.drone_rf.consts import BUI, M, Q
from rfml_uav.drone_rf.utils import get_segment_count


load_dotenv()

True

In [2]:
# Path of raw RF data
load_path = os.path.join(os.getenv("DATA_PATH"), "chunked")
save_path = os.path.join(os.getenv("DATA_PATH"), "PSD")
os.makedirs(save_path, exist_ok=True)

In [3]:
def get_psd_data(bui: str):
    data = []
    N = get_segment_count(bui)

    for n in tqdm(range(N)):
        x_chunks = np.load(os.path.join(load_path, f"{bui}L_{n}.npz"))['arr_0']
        y_chunks = np.load(os.path.join(load_path, f"{bui}H_{n}.npz"))['arr_0']

        for i in range(len(x_chunks)):
            xf = np.abs(fftshift(fft(x_chunks[i] - np.mean(x_chunks[i]), M)))[M//2:]
            yf = np.abs(fftshift(fft(y_chunks[i] - np.mean(y_chunks[i]), M)))[M//2:]
            c = np.mean(xf[-Q:]) / np.mean(yf[:Q]) # normalization factor
            segment_data = np.concatenate((xf, yf * c))
            data.append(segment_data)
    
    # Convert the list into a numpy array and square the data
    return np.array(data) ** 2

In [4]:
for bui in BUI:
    data = get_psd_data(bui)
    np.savetxt(os.path.join(save_path, f'{bui}.csv'), data, delimiter=',')

 76%|███████▌  | 16/21 [00:04<00:01,  3.31it/s]


KeyboardInterrupt: 